# W13 · Project kickoff / 專題起跑

**English.** The 12-week spine is done — you can now build the PEPS wrapper,
run the course protocols, quantize, and touch HIP. The capstone (W13–W14) turns that
into a small research contribution. **This week (W13):** pick one of four tracks,
rerun the *baseline you intend to compare*, and lock a success metric.
**Next week (W14):** run the extension and produce a notebook, nonblank CSV,
run manifest, submission manifest, and slide. Full brief: `docs/06_capstone.md`.

This notebook is a **scaffold**: the code cells are safe placeholders — you fill
in the `TODO(student)` parts for your chosen track.

**繁體中文.** 12 週主軸已完成 —— 你已能組 PEPS wrapper、執行協定、量化、碰 HIP。
專題(W13–W14)把這些變成一個小型研究貢獻。**本週(W13):**四選一,重跑要比較的
*基線*並鎖定成功指標。**下週(W14):**跑出延伸並產出完整可驗證 artifacts。
完整說明見 `docs/06_capstone.md`。此為**骨架**
notebook,程式格為安全佔位,請在 `TODO(student)` 處填入你選定軌道的內容。

In [ ]:
import sys, os; sys.path.insert(0, os.path.abspath('..'))
import math, torch, matplotlib.pyplot as plt
from peps.train import auto_device
def _required_text(value, name):
    if not isinstance(value, str) or not value.strip() or 'TODO' in value:
        raise ValueError(f'{name} must be nonblank and contain no TODO')
    return value.strip()
device = auto_device(); print('device', device)

## 1. Pick your track / 選擇軌道
Set `TRACK` below. Each track lists its starting files and the deliverable.
設定 `TRACK`;每條軌道列出起始檔案與交付物。

In [ ]:
# The four capstone tracks (see docs/06_capstone.md for the full brief + rubric).
TRACKS = {
  'a': {'name': 'Short 3D video volume (x, y, t)',
        'start': ['peps/wrapper.py', 'peps/projector.py', 'peps/encoders/grid.py',
                  'apps/image/ (as a template)', 'peps/train.py'],
        'deliverable': '3D grid vs 3D Grid-PEPS on one small licensed clip',
        'csv': 'results/capstone_video3d.csv'},
  'b': {'name': 'Quantization calibration or short QAT recovery',
        'start': ['peps/quant/ptq.py', 'notebooks/W10_quantization.ipynb',
                  'tests/test_quantization.py'],
        'deliverable': 'clipping/calibration or QAT vs rerun per-channel PTQ',
        'csv': 'results/capstone_quant_calibration.csv'},
  'c': {'name': 'Design + evaluate a new aggregator (beyond concat/pink/brownian)',
        'start': ['peps/aggregate.py', 'apps/image/build.py',
                  'notebooks/W06_pink_peps.ipynb'],
        'deliverable': 'new aggregator kind + params-vs-PSNR vs the existing three',
        'csv': 'results/capstone_aggregator.csv'},
  'd': {'name': 'End-to-end PEPS runtime optimization and receipt',
        'start': ['hip/wmma_mlp.hip', 'hip/fused_peps_kernel.hip',
                  'hip/bench_latency.sh', 'tests/test_hip_parity.py'],
        'deliverable': 'one optimization vs rerun full-pipeline baseline; parity + latency',
        'csv': 'results/capstone_runtime.csv'},
}

# >>> Pick your track here <<<
TRACK = 'a'  # TODO(student): one of 'a' | 'b' | 'c' | 'd'
t = TRACKS[TRACK]
print('Track', TRACK, '-', t['name'])
print('Starting files:'); [print('  -', s) for s in t['start']]
print('Deliverable   :', t['deliverable'])
print('Results CSV   :', t['csv'])

## 2. Rerun the matched baseline / 重跑公平基線
Tracked result CSVs are legacy-unverified and may be used only for orientation.
Rerun the baseline under your selected profile; copied numbers do not pass.
Fill the branch for your track; the others are guidance.

既有結果 CSV 皆為 legacy-unverified,只能作為方向。請在所選 profile 下親自重跑
基線;抄數字不能通過。

In [ ]:
# TODO(student): rerun ONE baseline for your track. Sketches below.
if TRACK == 'a':
    # Use a self-created/CC0 clip, e.g. 8-16 frames at <=64x64.
    # GridEncoder already supports dim=3; coordinates are (x, y, t).
    print('a) baseline = plain 3D grid on the exact clip used by the extension')
elif TRACK == 'b':
    # int4/int8, mixed precision, per-channel PTQ, and metadata accounting exist.
    # Orientation only: tracked per-channel means are 38.702/41.623 dB at
    # 11.556/11.564 bpp; rerun them, do not copy these legacy-unverified rows.
    print('b) baseline = rerun per-channel PTQ at the chosen total encoded bits')
elif TRACK == 'c':
    # Rerun corrected concat/pink under one matched image profile.
    print('c) baseline = rerun concat/pink under the same data, seeds, and budget')
elif TRACK == 'd':
    # Hardware-gated: rerun the complete pipeline, not a standalone GEMM CSV row.
    print('d) baseline = rerun full-pipeline parity + latency with a hardware receipt')
else:
    raise ValueError(f'unknown TRACK {TRACK!r} — pick a/b/c/d')

## 3. Lock a success metric + record the baseline / 鎖定指標並記錄基線
State the single number that defines success and save the verified rerun.
The helper refuses blank, non-finite, unknown-profile, or legacy values.

寫下定義成功的單一數字並儲存已驗證的重跑。helper 會拒絕空白、非有限值、
未知 profile 或 legacy 狀態。

In [ ]:
import csv
os.makedirs('../results', exist_ok=True)
PROFILE = 'course_fast'  # TODO(student): course_fast or paper_exact
SEED = 0
def record_baseline(metric_name, value, units, path=None):
    metric_name = _required_text(metric_name, 'metric_name')
    units = _required_text(units, 'units')
    if PROFILE not in {'course_fast', 'paper_exact'}:
        raise ValueError('PROFILE must be course_fast or paper_exact')
    value = float(value)
    if not math.isfinite(value): raise ValueError('baseline value must be finite')
    if isinstance(SEED, bool) or not isinstance(SEED, int): raise ValueError('SEED must be int')
    path = path or f'../results/capstone_{TRACK}_baseline.csv'
    with open(path, 'w', newline='') as f:
        w = csv.writer(f, lineterminator='\n')
        w.writerow(['stage', 'metric', 'value', 'units', 'profile', 'seed', 'status'])
        w.writerow(['baseline', metric_name, value, units, PROFILE, SEED, 'verified'])
    print('wrote', path)
    return path

# No example result is supplied: copying a legacy number is not a rerun.
# record_baseline('YOUR_METRIC', YOUR_FINITE_VALUE, 'YOUR_UNITS')
print('After your rerun, call record_baseline(...) to save verified evidence.')

## 4. Deliverables & timeline / 交付物與時程
Every track ships the same evidence bundle:

- an executed **notebook** and nonblank numeric **results CSV**,
- a **run manifest** and completed **capstone submission JSON**,
- one bilingual **Marp slide** that passes the course build.

**W13 exit check:** track chosen, baseline rerun, metric + verified CSV in place.

每條軌道都交付 notebook、非空數值 CSV、run/submission manifest 與一張可建置的
雙語 Marp 投影片。**W13 出關檢查:**已選題、已重跑基線、指標與 verified CSV 就緒。

## 5. Kickoff takeaway / 起跑小結
A good capstone is **narrow and honest**: one baseline, one change, one number,
reported truthfully — win or lose. Next week you execute and present.

好的專題**窄而誠實**:一個基線、一個改動、一個數字,如實回報 —— 無論勝負。
下週執行並發表。